<a href="https://colab.research.google.com/github/xyt556/I-GUIDE-GeoAI-Education/blob/main/notebooks/11-sam-geospatial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 地理空间领域的 Segment Anything

## 介绍

传统的遥感图像分割需要收集标注的训练数据，训练模型，并期望它能泛化到新的场景。每个新的地理区域、传感器或季节都可能意味着需要重新开始标注过程。

Meta AI 于 2023 年 4 月发布的 [Segment Anything Model](https://ai.meta.com/datasets/segment-anything) (SAM) 改变了这一局面，它实现了零样本分割——无需任务特定训练即可分割图像中的几乎任何对象。最新一代的 SAM 3 在此基础上进行了增强，提供了更出色的分割质量、多模态提示支持以及用于跨视频帧跟踪对象的流式记忆架构。

一个模型可以描绘内罗毕的建筑足迹，绘制爱荷华州的农田地图，并追踪亚马逊的河流网络，而无需查看任何这些地区的单个标注示例。虽然 SAM 3 不会取代领域专业知识，但它极大地加速了许多地理空间工作流程中最耗费人力的阶段。

`segment-geospatial` Python 包 (`samgeo`) 通过 `SamGeo3` 和 `SamGeo3Video` 类将 SAM 3 与地理空间世界连接起来。这些类处理地理参考输入、坐标系统和矢量输出格式，以便预测结果直接集成到 GIS 工作流程中。

本教程将介绍 `samgeo` 为 SAM 3 提供的实用工作流程，包括文本、点和框提示；分块和批量分割；交互式分割；以及视频对象跟踪。

## 学习目标

完成本教程后，您将能够：

- 解释 SAM 3 如何实现零样本图像和视频分割
- 使用 `SamGeo3` 通过文本、点和框提示分割卫星图像
- 将带有置信度分数的分割掩膜保存为地理参考的 GeoTIFF 文件
- 使用共享工作流程批量处理多个图像
- 使用分块方法分割大型 GeoTIFF 图像
- 使用内置的地图界面交互式分割图像
- 使用 `SamGeo3Video` 跟踪视频帧中的对象

## SAM 3 工作原理

SAM 3 的架构由三个核心图像分割组件组成，此外还有一个用于视频的流式记忆模块。

**图像编码器。** 视觉 Transformer (ViT) 将输入图像处理成高维特征嵌入。这个昂贵的步骤每个图像只运行一次，并且对未见过的图像（包括卫星和航空照片）具有良好的泛化能力。

**提示编码器。** 提示编码器将用户提供的提示（文本描述、点坐标或边界框）转换为模型可以与图像嵌入一起使用的格式。它很轻量级，因此可以针对相同的缓存嵌入快速测试不同的提示。

**掩膜解码器。** 掩膜解码器结合图像嵌入和编码提示，使用带有交叉注意力的修改版 Transformer 解码器生成最终的分割掩膜和置信度分数。

**视频流式记忆。** SAM 3 维护一个包含先前看到的帧及其分割结果的内存库。当新帧到达时，内存库会提供分割信息，以便在数百帧中连贯地跟踪对象。

这种设计意味着昂贵的图像编码只发生一次，然后可以廉价地评估任意数量的提示针对缓存的嵌入。

In [ ]:
# %pip install -U "geoai-py[extra]" "segment-geospatial[samgeo3]"

In [ ]:
import os
import geoai
import leafmap
from samgeo import SamGeo3, SamGeo3Video, download_file, show_image
from samgeo.common import raster_to_vector, regularize

SAM 3 首次使用前需要通过 Hugging Face 获得访问批准。请在 [SAM 3 模型页面](https://huggingface.co/facebook/sam3) 请求访问权限。获得访问权限后，请使用 Hugging Face 登录：

In [ ]:
# from huggingface_hub import login
# login()

## 图像分割

核心工作流程遵循三个步骤：加载图像，通过提示生成掩膜，并保存结果。

下载一张涵盖加州大学伯克利分校的卫星图像示例：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
image_path = download_file(url)

在交互式地图上显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化 `SamGeo3` 并加载图像。`set_image()` 方法只运行一次图像编码器并缓存嵌入：

In [ ]:
sam3 = SamGeo3(backend="meta", device=None, checkpoint_path=None, load_from_HF=True)
sam3.set_image(image_path)

### 文本提示分割

使用自然语言文本提示生成掩膜：

In [ ]:
sam3.generate_masks(prompt="building")

将检测到的对象可视化为原始图像上的彩色叠加层：

In [ ]:
sam3.show_anns()

使用 `show_masks()` 仅在空白背景上显示分割掩膜：

In [ ]:
sam3.show_masks()

将掩膜保存为 GeoTIFF。设置 `unique=True` 会为每个分割对象分配一个不同的整数值：

In [ ]:
sam3.save_masks(output="building_masks.tif", unique=True)

使用 `save_scores` 参数也可以捕获预测置信度：

In [ ]:
sam3.save_masks(
    output="building_masks_with_scores.tif",
    save_scores="building_scores.tif",
    unique=True,
)

按置信度分数显示彩色掩膜：

In [ ]:
sam3.show_masks(cmap="coolwarm")

在交互式地图上可视化置信度分数：

In [ ]:
m.add_raster("building_masks.tif", layer_name="Building masks", visible=False)
m.add_raster(
    "building_scores.tif",
    layer_name="Building scores",
    cmap="coolwarm",
    opacity=0.8,
    nodata=0,
    vmin=0.5,
    vmax=1.0,
)
m.add_colormap(cmap="coolwarm", vmin=0.5, vmax=1.0, label="Confidence score")
m

### 框提示分割

边界框提示告诉 SAM 3 使用框内的对象作为参考，并在图像的其他地方搜索相似的对象。以 `[xmin, ymin, xmax, ymax]` 格式使用地理坐标指定框：

In [ ]:
# Define boxes in [xmin, ymin, xmax, ymax] format
boxes = [[-122.2597, 37.8709, -122.2587, 37.8717]]
box_labels = [True]  # True=include, False=exclude

sam3.generate_masks_by_boxes(boxes, box_labels, box_crs="EPSG:4326")

在图像上叠加边界框：

In [ ]:
sam3.show_boxes(boxes, box_labels, box_crs="EPSG:4326")

可视化分割掩膜：

In [ ]:
sam3.show_anns()

将框提示分割掩膜保存为地理参考的 GeoTIFF：

In [ ]:
building_mask_path = "building_masks.tif"
sam3.save_masks(output=building_mask_path, unique=True)

在原始卫星图像上叠加掩膜：

In [ ]:
geoai.view_raster(building_mask_path, nodata=0, opacity=0.7, basemap=image_path)

在进入下一节之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

## 用于实例分割的点提示

点提示使用前景点（label=1）和背景点（label=0）来分割特定对象。通过 `enable_inst_interactivity=True` 初始化 `SamGeo3` 以启用此模式。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/truck-example.jpg"
image_path = download_file(url)

显示带有轴标签的图像，以便读取像素坐标：

In [ ]:
show_image(image_path, axis="on")

初始化带有实例交互性的 SAM 3 并加载图像：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

### 单点提示

以 `[x, y]` 像素坐标指定单个前景点：

In [ ]:
sam.generate_masks_by_points([[750, 370]])

在图像上叠加点标记：

In [ ]:
sam.show_points([[750, 370]], [1])

可视化结果分割掩膜：

In [ ]:
sam.show_anns()

将掩膜保存为 PNG 文件：

In [ ]:
sam.save_masks("truck_mask.png", unique=True)

### 多个点

同一对象上的多个前景点可以改善大型或不规则形状对象的覆盖范围：

In [ ]:
sam.generate_masks_by_points([[500, 375], [1125, 625]], point_labels=[1, 1])

叠加两个点标记：

In [ ]:
sam.show_points([[500, 375], [1125, 625]], [1, 1])

可视化结果分割掩膜：

In [ ]:
sam.show_anns()

### 背景点

背景点（label=0）从掩膜中排除一个区域，以细化模糊的边界：

In [ ]:
sam.generate_masks_by_points([[750, 370], [1125, 625]], point_labels=[1, 0])

叠加前景（绿色）和背景（红色）点标记：

In [ ]:
sam.show_points([[750, 370], [1125, 625]], [1, 0])

可视化精细化的分割掩膜：

In [ ]:
sam.show_anns()

### 多个框提示

可以同时处理多个框以在一个调用中分割多个对象：

In [ ]:
boxes = [
    [75, 275, 1725, 850],  # Whole truck
    [425, 600, 700, 875],  # Rear wheel
    [1375, 550, 1650, 800],  # Front wheel on the passenger side
    [1240, 675, 1400, 750],  # Front wheel on the driver's side
]
sam.generate_masks_by_boxes_inst(boxes)

在图像上叠加边界框：

In [ ]:
sam.show_boxes(boxes)

可视化分割掩膜：

In [ ]:
sam.show_anns()

保存多框分割掩膜：

In [ ]:
sam.save_masks("truck_boxes_mask.png", unique=True)

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

### 地理空间数据的批量点提示

对于地理参考图像，提供地理坐标作为点提示。`generate_masks_by_points_patch()` 方法会分割每个点处的对象并保存地理参考结果。

下载卫星图像和建筑物质心 GeoJSON：

In [ ]:
image_url = "https://data.source.coop/opengeos/geoai/wa-building-image.tif"
geojson_url = "https://data.source.coop/opengeos/geoai/wa-building-centroids.geojson"
image_path = download_file(image_url)
geojson_path = download_file(geojson_url)

显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化 SAM 3 并设置图像：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

在每个点位置分割建筑物：

In [ ]:
point_coords_batch = [
    [-117.599896, 47.655345],
    [-117.59992, 47.655167],
    [-117.599928, 47.654974],
    [-117.599518, 47.655337],
]

sam.generate_masks_by_points_patch(
    point_coords_batch=point_coords_batch,
    point_crs="EPSG:4326",
    output="masks.tif",
    dtype="uint8",
)

在图像上叠加点标记：

In [ ]:
sam.show_points(point_coords_batch, point_crs="EPSG:4326")

将掩膜添加到交互式地图：

In [ ]:
m.add_raster("masks.tif", cmap="viridis", nodata=0, opacity=0.7, layer_name="Mask")
m

您也可以直接提供包含点几何体的 GeoJSON 文件：

In [ ]:
sam.generate_masks_by_points_patch(
    point_coords_batch=geojson_path,
    point_crs="EPSG:4326",
    output="building_masks.tif",
    dtype="uint16",
)

将建筑物掩膜和质心标记添加到地图：

In [ ]:
m.add_raster(
    "building_masks.tif", cmap="jet", nodata=0, opacity=0.7, layer_name="Building masks"
)
m.add_circle_markers_from_xy(
    geojson_path, radius=3, color="red", fill_color="yellow", fill_opacity=0.8
)
m

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

## 建筑物提取的框提示

框提示非常适合建筑物足迹，因为建筑物具有明确的矩形范围。本节演示从分割到矢量导出和正则化的完整工作流程。

显示卫星图像：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Satellite image")
m

初始化 SAM 3：

In [ ]:
sam = SamGeo3(backend="meta", enable_inst_interactivity=True)
sam.set_image(image_path)

在地理坐标中定义建筑物周围的边界框：

In [ ]:
if m.user_rois is not None:
    boxes = m.user_rois
else:
    boxes = [
        [-117.5995, 47.6518, -117.5988, 47.652],
        [-117.5987, 47.6518, -117.5979, 47.652],
    ]

使用边界框生成掩膜：

In [ ]:
sam.generate_masks_by_boxes_inst(boxes=boxes, box_crs="EPSG:4326")

将建筑物掩膜保存为 GeoTIFF：

In [ ]:
sam.save_masks(output="mask.tif", dtype="uint8")

在地图上叠加掩膜：

In [ ]:
m.add_raster("mask.tif", cmap="viridis", nodata=0, opacity=0.5, layer_name="Mask")
m

### 使用矢量文件作为框提示

而不是指定单个坐标，直接传递 GeoJSON 或 Shapefile：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/wa-building-bboxes.geojson"
geojson_path = download_file(url)

在卫星图像上叠加边界框：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Image")
style = {
    "color": "#ffff00",
    "weight": 2,
    "fillColor": "#7c4185",
    "fillOpacity": 0,
}
m.add_vector(geojson_path, style=style, zoom_to_layer=True, layer_name="Bboxes")
m

使用 GeoJSON 文件作为框提示，为所有建筑物生成掩膜：

In [ ]:
output_masks = "building_masks.tif"
sam.generate_masks_by_boxes_inst(
    boxes=geojson_path,
    box_crs="EPSG:4326",
    output=output_masks,
    dtype="uint16",
    multimask_output=False,
)

显示建筑物掩膜：

In [ ]:
m.add_raster(
    output_masks, cmap="jet", nodata=0, opacity=0.5, layer_name="Building masks"
)
m

### 转换为矢量并正则化

将栅格掩膜转换为矢量多边形：

In [ ]:
output_vector = "building_vector.geojson"
raster_to_vector(output_masks, output_vector)

`regularize()` 函数调整多边形几何形状以生成更清晰、更规则的足迹：

In [ ]:
output_regularized = "building_regularized.geojson"
regularize(output_vector, output_regularized)

将正则化的足迹添加到地图：

In [ ]:
m.add_vector(
    output_regularized, style=style, layer_name="Building regularized", info_mode=None
)
m

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

## 批量分割

批量分割通过单个工作流程处理具有相同提示的多个图像。

下载涵盖加州大学伯克利分校校园的四张卫星图像瓦片：

In [ ]:
image_paths = []
for i in range(1, 5):
    url = f"https://data.source.coop/opengeos/geoai/uc-berkeley-{i}.tif"
    image_path = download_file(url)
    image_paths.append(image_path)

在交互式地图上显示所有四个瓦片：

In [ ]:
m = leafmap.Map()
for i, image_path in enumerate(image_paths):
    m.add_raster(image_path, layer_name=f"image_{i + 1}")
m

初始化 SAM 3 并批量加载所有图像：

In [ ]:
sam3 = SamGeo3(backend="meta", device=None, checkpoint_path=None, load_from_HF=True)
sam3.set_image_batch(image_paths)

使用单个文本提示为所有图像生成掩膜：

In [ ]:
sam3.generate_masks_batch("building", min_size=100)

```text
处理了 4 张图像，共找到 174 个对象。
```

检查每张图像中检测到的对象数量：

In [ ]:
for i, result in enumerate(sam3.batch_results):
    print(f"Image {i + 1}: Found {len(result['masks'])} objects")

```text
图像 1：找到 46 个对象
图像 2：找到 50 个对象
图像 3：找到 64 个对象
图像 4：找到 14 个对象
```

以网格布局可视化所有结果：

In [ ]:
sam3.show_anns_batch(ncols=2, show_bbox=True, show_score=True, figsize=(12, 8))

将注释图像保存到磁盘：

In [ ]:
sam3.show_anns_batch(output_dir="output/annotations/", prefix="ann", dpi=300)

将每个图像的掩膜导出为单独的地理参考 GeoTIFF：

In [ ]:
saved_files = sam3.save_masks_batch(
    output_dir="output/", prefix="building_mask", unique=True
)

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

## 大图像的分块分割

大型卫星图像通常会超出 GPU 内存限制。`generate_masks_tiled()` 方法将图像分割成重叠的瓦片，独立处理每个瓦片，然后合并结果。

下载用于水体检测的大型 NAIP 图像：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/naip_water_train.tif"
image_path = download_file(url)

检查图像尺寸：

In [ ]:
geoai.print_raster_info(image_path)

初始化 SAM 3 并运行分块分割：

In [ ]:
sam = SamGeo3(backend="meta")

output_path = "segmentation_mask.tif"

sam.generate_masks_tiled(
    source=image_path,
    prompt="water",
    output=output_path,
    tile_size=1024,
    overlap=128,
    min_size=100,
    unique=False,
    dtype="int32",
    verbose=True,
)

可视化水体分割结果：

In [ ]:
m = leafmap.Map()
m.add_raster(image_path, layer_name="Original Image")
m.add_raster(
    output_path, nodata=0, opacity=0.8, cmap="Blues", layer_name="Segmentation Mask"
)
m

将栅格掩膜转换为矢量多边形：

In [ ]:
vector_path = "segmentation_mask.gpkg"
geoai.raster_to_vector(output_path, vector_path)

`regularize()` 函数调整多边形几何形状以生成更清晰、更规则的足迹：

In [ ]:
smooth_vector_path = "segmentation_mask_smooth.gpkg"
gdf = geoai.smooth_vector(vector_path, smooth_vector_path)

将平滑的多边形添加到地图：

In [ ]:
style = {
    "color": "#ff0000",
    "weight": 2,
    "fillOpacity": 0,
}
m.add_gdf(gdf, layer_name="Smoothed Vector", info_mode=None, style=style)
m

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

在调整分块分割时，请考虑以下参数：

- **tile_size**：较大的瓦片捕获更多上下文，但需要更多 GPU 内存。从 1024 开始。
- **overlap**：较高的重叠（128-256 像素）可以防止边界伪影，但会增加处理时间。
- **min_size / max_size**：使用这些参数过滤掉噪声或不相关的较大区域。
- **dtype**：对于许多对象，使用 `int32`；对于多达 65,535 个对象，使用 `uint16`；或者对于二值掩膜，使用 `uint8`。

## 交互式分割

`show_map()` 方法启动一个交互式 Jupyter 小部件，将地图界面与 SAM 3 分割结合起来。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/uc-berkeley.tif"
image_path = download_file(url)

使用 Hugging Face Transformers 后端初始化 SAM 3：

In [ ]:
sam3 = SamGeo3(
    backend="transformers", device=None, checkpoint_path=None, load_from_HF=True
)
sam3.set_image(image_path)

生成一组初始掩膜：

In [ ]:
sam3.generate_masks(prompt="building")
sam3.save_masks("masks.tif")

启动交互式界面。输入文本提示或绘制一个矩形，然后点击 **Segment**：

In [ ]:
sam3.show_map(height="700px", min_size=10)

该界面支持文本提示模式和边界框模式。结果会更新，而无需重新运行图像编码器。

继续之前释放 GPU 内存：

In [ ]:
geoai.empty_cache()

## 视频分割

`SamGeo3Video` 将 SAM 3 扩展到视频，支持 MP4 文件、JPEG 帧目录和 GeoTIFF 文件目录。

### 文本提示视频分割

下载汽车视频样本：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/cars.mp4"
video_path = download_file(url)

初始化 `SamGeo3Video`，加载视频并预览：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.show_video(video_path)

使用文本提示生成掩膜：

In [ ]:
sam.generate_masks("car")

显示第一帧以验证检测结果：

In [ ]:
sam.show_frame(0, axis="on")

显示每 20 帧采样一次的帧网格：

In [ ]:
sam.show_frames(frame_stride=20, ncols=3)

通过 ID 移除不需要的对象并重新传播：

In [ ]:
sam.remove_object(2)
sam.propagate()
sam.show_frame(0)

保存每帧掩膜：

In [ ]:
os.makedirs("output", exist_ok=True)
sam.save_masks("output/masks")

渲染带注释的视频：

In [ ]:
sam.save_video("output/segmented.mp4", fps=25)

关闭视频会话：

In [ ]:
sam.close()

### 点提示视频分割

点提示可以精确控制要跟踪的对象。

初始化新的视频会话：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.init_tracker()
sam.show_frame(0, axis="on")

为每个对象添加点提示并传播：

In [ ]:
sam.add_point_prompts([[300, 200]], [1], obj_id=1, frame_idx=0)
sam.add_point_prompts([[420, 200]], [1], obj_id=2, frame_idx=0)
sam.propagate()

显示带有跟踪掩膜的第一帧：

In [ ]:
sam.show_frame(0, axis="on")

可视化采样帧的跟踪情况：

In [ ]:
sam.show_frames(frame_stride=20, ncols=3)

同时使用正点和负点来细化掩膜：

In [ ]:
# Positive point on windshield, negative point on car body
sam.add_point_prompts(
    points=[[335, 195], [335, 220]],
    labels=[1, 0],
    obj_id=1,
    frame_idx=0,
)
sam.propagate()
sam.show_frames(frame_stride=20, ncols=3)

保存细化结果并关闭会话：

In [ ]:
sam.save_masks("output/masks")
sam.save_video("output/segmented.mp4", fps=25)
sam.close()

### 对象跟踪

SAM 3 可以在视频帧中跟踪多个对象，这对于监控车辆、球员和其他移动目标很有用。

下载篮球视频：

In [ ]:
url = "https://data.source.coop/opengeos/geoai/basketball.mp4"
video_path = download_file(url)

初始化并预览视频：

In [ ]:
sam = SamGeo3Video()
sam.set_video(video_path)
sam.show_video(video_path)

用文本提示检测并跟踪所有球员：

In [ ]:
sam.generate_masks("player")

创建显示名称标签并可视化第一帧：

In [ ]:
player_names = {}
for i in range(15):
    player_names[i] = f"Player {i}"
sam.show_frame(0, axis="on", show_ids=player_names)

移除虚假检测：

In [ ]:
sam.remove_object(obj_id=[5, 8, 12, 13])
sam.propagate()
sam.show_frame(0, show_ids=player_names)

保存掩膜，渲染带注释的视频，并预览：

In [ ]:
os.makedirs("output", exist_ok=True)
sam.save_masks("output/masks")
sam.save_video("output/players_segmented.mp4", fps=60, show_ids=player_names)
sam.show_video("output/players_segmented.mp4")

关闭会话并释放 GPU 内存：

In [ ]:
sam.close()
sam.shutdown()

## 关键要点

1. SAM 3 实现了地理空间图像和视频的零样本分割，无需任务特定的训练数据。

2. 文本、点和框提示满足不同的需求，从可访问的自然语言查询到精确的空间控制。

3. 置信度分数允许您在下游处理中过滤低质量预测。

4. 批量和分块分割将 SAM 3 扩展到操作规模和超出 GPU 内存的大图像。

5. `show_map()` 交互式界面无需编写代码即可实现快速探索性分析。

6. `SamGeo3Video` 使用流式记忆架构在帧之间连贯地跟踪对象。

7. 整个管道中都保留了地理参考，因此输出可直接用于 GIS 软件。